# Phase 1 runner (Colab T4)

Thin executor. It pulls the repo and runs one script on the T4. Develop on the laptop, push to GitHub, then run the cells here top to bottom.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

You only ever edit the **config cell**. Everything below it is fixed.

## 0. Check the GPU is a T4

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 1. Config: pick what to run

This is the only cell you change. Pick the script and its arguments. Re-run from here down each time.

In [ ]:
REPO_URL = "https://github.com/bilalrana351/llm-inference-optimization.git"
BRANCH   = "main"

# The command to run, relative to the repo root. Switch this line to change task.
#
#   Task 1  baseline:  python scripts/baseline_hf.py --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 256
#   Task 2  vllm:      python scripts/bench_vllm.py  --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 256
#   Task 3  oom sweep: python scripts/oom_sweep.py   --model Qwen/Qwen2.5-1.5B
RUN_CMD = "python scripts/baseline_hf.py --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 256"

# vLLM is only needed for Task 2. Leave False for the baseline and OOM sweep so
# you do not pay its install time on every run.
INSTALL_VLLM = False
VLLM_VERSION = "0.6.1.post2"

## 2. Pull the repo (clone first time, fast-forward after)

In [ ]:
import os, subprocess

REPO_DIR = "/content/llm-inference-optimization"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(REPO_DIR)
print("at", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"])

## 3. Install dependencies

Baseline deps are quick. vLLM is the slow one and only installs when `INSTALL_VLLM` is True.

In [ ]:
!pip install -q -r requirements.txt
if INSTALL_VLLM:
    !pip install -q vllm=={VLLM_VERSION}
    print("vllm installed; you may need to restart the runtime if torch was upgraded")

## 4. Run

The script writes a CSV row (and plots, for the OOM sweep) into `results/`. Paste any traceback back into the development session to debug.

In [ ]:
import shlex, subprocess
subprocess.run(shlex.split(RUN_CMD), check=True)

## 5. Inspect results

Show the logged rows and any plots produced.

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display

for csv in sorted(glob.glob("results/*.csv")):
    print("#", csv)
    display(pd.read_csv(csv))

for png in sorted(glob.glob("results/*.png")):
    print("#", png)
    display(Image(png))

## 6. (Optional) Commit results back

Results are part of the artifact. To push CSVs and plots from Colab, set a token and run this. Skip it if you would rather download `results/` and commit from the laptop.

In [ ]:
# from getpass import getpass
# token = getpass("GitHub token: ")
# !git config user.email "bilalrana351@gmail.com"
# !git config user.name "bilalrana351"
# !git add results/ blog/
# !git commit -m "results: add run from Colab T4"
# !git push https://{token}@github.com/bilalrana351/llm-inference-optimization.git {BRANCH}